In [1]:
import os
PROJECT_PATH = "/mnt/c/Users/Yeray/Desktop/MUIA/Computer Vision/COMPUTERVISIONPROYECT"
XVIEW_RECOGNITION_PATH = os.path.join(PROJECT_PATH, "data/xview_detection")
DATASET_NAME = 'stratified'

LOG_NAME = '_'.join([DATASET_NAME,'log.json'])
TRAIN_NAME = '_'.join([DATASET_NAME,'train.tfrecord'])
VALIDATION_NAME = '_'.join([DATASET_NAME,'validation.tfrecord'])

In [7]:
import tensorflow as tf
import numpy as np
import os
import json
import uuid
from tqdm import tqdm
from sklearn.model_selection import train_test_split

JSON_FILE = os.path.join(XVIEW_RECOGNITION_PATH, 'xview_det_train.json')

# Mapping categories to IDs
categories = {
    0: 'Small car', 1: 'Bus', 2: 'Truck', 3: 'Building'
}
id_to_name = {v: k for k, v in categories.items()}

In [8]:
id_to_name

{'Small car': 0, 'Bus': 1, 'Truck': 2, 'Building': 3}

In [9]:
# -----------------------------
# Generic classes
# -----------------------------
class GenericObject:
    def __init__(self):
        self.id = uuid.uuid4()
        self.bb = (-1, -1, -1, -1)
        self.category = -1

class GenericImage:
    def __init__(self, filename, width, height):
        self.filename = filename
        self.width = width
        self.height = height
        self.objects = []

    def add_object(self, obj: GenericObject):
        self.objects.append(obj)

In [16]:
with open(JSON_FILE) as f:
    json_data = json.load(f)

    print(json_data.keys())
    print(f"Num images: {len(json_data['images'])}")
    print(f"Num annotations: {len(json_data['annotations'])}")
    print(f"Num categories: {len(json_data['categories'])}")
    print(f"json_data['images']['0']: {json_data['images']['0']}")
    print(f"json_data['annotations']['0']: {json_data['annotations']['0']}")
    print(f"json_data['categories']['0']: {json_data['categories']['0']}")

dict_keys(['info', 'images', 'annotations', 'categories'])
Num images: 7606
Num annotations: 481112
Num categories: 4
json_data['images']['0']: {'image_id': '2511_66439541-a371-4b68-93c8-8c62da2cd64e.tif', 'filename': 'xview_train/2511_66439541-a371-4b68-93c8-8c62da2cd64e.tif', 'num_objects': 9, 'width': 640, 'height': 640}
json_data['annotations']['0']: {'image_id': '2511_66439541-a371-4b68-93c8-8c62da2cd64e.tif', 'category_id': 'Building', 'bbox': [562, 489, 663, 517]}
json_data['categories']['0']: {'id': 18, 'name': 'Small car', 'supercategory': 'Passenger vehicle'}


In [18]:
import json
import os
import random
import shutil

JSON_PATH = "data/xview_detection/xview_det_train.json"
IMAGE_ROOT = "data/xview_detection"
OUT_ROOT = "data/dataset_yolo"
TRAIN_RATIO = 0.8

# -------------------------
# Load JSON
# -------------------------
with open(JSON_PATH) as f:
    data = json.load(f)

images = data["images"]
annotations = data["annotations"]
categories = data["categories"]

print("Images:", len(images))
print("Annotations:", len(annotations))
print("Categories:", len(categories))

# -------------------------
# Class mapping
# -------------------------
class_names = sorted({c["name"] for c in categories.values()})
class_to_id = {name: i for i, name in enumerate(class_names)}
print("Class map:", class_to_id)

# -------------------------
# Image metadata
# -------------------------
image_info = {
    img["image_id"]: img
    for img in images.values()
}

# -------------------------
# Group annotations
# -------------------------
by_image = {}

for ann in annotations.values():
    img_id = ann["image_id"]
    cls = class_to_id[ann["category_id"]]
    xmin, ymin, xmax, ymax = ann["bbox"]

    by_image.setdefault(img_id, []).append(
        (cls, xmin, ymin, xmax, ymax)
    )

print("Images with annotations:", len(by_image))

# -------------------------
# Train / val split
# -------------------------
image_ids = list(by_image.keys())
random.shuffle(image_ids)

split = int(len(image_ids) * TRAIN_RATIO)
train_ids = image_ids[:split]
val_ids = image_ids[split:]

# -------------------------
# Output dirs
# -------------------------
for s in ["train", "val"]:
    os.makedirs(f"{OUT_ROOT}/images/{s}", exist_ok=True)
    os.makedirs(f"{OUT_ROOT}/labels/{s}", exist_ok=True)

# -------------------------
# Write YOLO files
# -------------------------
def write_split(ids, split):
    for img_id in ids:
        info = image_info[img_id]
        w, h = info["width"], info["height"]

        src_img = os.path.join(IMAGE_ROOT, info["filename"])
        dst_img = f"{OUT_ROOT}/images/{split}/{img_id}"
        label_path = f"{OUT_ROOT}/labels/{split}/{img_id.replace('.tif','.txt')}"

        with open(label_path, "w") as f:
            for cls, xmin, ymin, xmax, ymax in by_image[img_id]:
                x = ((xmin + xmax) / 2) / w
                y = ((ymin + ymax) / 2) / h
                bw = (xmax - xmin) / w
                bh = (ymax - ymin) / h
                f.write(f"{cls} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}\n")

        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)

# -------------------------
# Run
# -------------------------
write_split(train_ids, "train")
write_split(val_ids, "val")

print("✅ Conversion finished")



Images: 7606
Annotations: 481112
Categories: 4
Class map: {'Building': 0, 'Bus': 1, 'Small car': 2, 'Truck': 3}
Images with annotations: 7606
✅ Conversion finished


In [ ]:
pip install ultralytics

In [ ]:
!yolo detect train model=yolov8n.pt data=xview.yaml imgsz=640 epochs=50 batch=8

In [15]:
import os
from ultralytics import YOLO

# Paths
TEST_DIR = "data/xview_detection/xview_test"
MODEL_PATH = "runs/detect/train/weights/best.pt"
OUTPUT_JSON = "experiment_results/Detection/1_YOLO/prediction.json"

# Load trained YOLOv8 model
model = YOLO(MODEL_PATH)

# Categories (must match the training class names)
categories = {
    0: "Building",
    1: "Bus",
    2: "Small car",
    3: "Truck"
}

# Prepare JSON structure
predictions_data = {"images": {}, "annotations": {}, "categories": categories}
imgs_idx, annos_idx = 0, 0

# Iterate over test images
for img_file in os.listdir(TEST_DIR):
    if not img_file.lower().endswith((".jpg", ".png", ".tif")):
        continue

    img_path = os.path.join(TEST_DIR, img_file)
    
    # Run prediction
    results = model.predict(img_path, verbose=False)
    
    # YOLOv8 returns a list of results (one per image), we have only 1 image
    result = results[0]  
    boxes = result.boxes.xyxy.cpu().numpy()        # [xmin, ymin, xmax, ymax]
    scores = result.boxes.conf.cpu().numpy()      # confidence
    class_ids = result.boxes.cls.cpu().numpy().astype(int)

    # Fill images JSON
    image_data = {
        "image_id": img_file,
        "filename": "xview_test/"+img_path.split('/')[-1],
        "num_objects": len(boxes),
        "width": int(result.orig_shape[1]),
        "height": int(result.orig_shape[0])
    }
    predictions_data["images"][imgs_idx] = image_data
    imgs_idx += 1

    # Fill annotations
    for i in range(len(boxes)):
        bbox = boxes[i]
        cls = class_ids[i]
        conf = float(scores[i])
        annotation_data = {
            "image_id": img_file,
            "category_id": categories[cls],
            "bbox": [int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])],
            "confidence": conf
        }
        predictions_data["annotations"][annos_idx] = annotation_data
        annos_idx += 1

# Save predictions
import json
with open(OUTPUT_JSON, "w") as f:
    json.dump(predictions_data, f)

print(f"✅ Predictions saved to {OUTPUT_JSON}")

✅ Predictions saved to experiment_results/Detection/1_YOLO/prediction.json
